# 03 — Historical Diabetes experiment

This notebook preserves a path to the original repository experiment artifacts. It is separate from the new generic data pipeline. The legacy data already contains a prepared `full` / `quasi` mask configuration.

In [ ]:
from pathlib import Path
import sys
import torch
import pandas as pd

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / "run_experiment.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or its notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
from pmae_tabular.experiments import _namespace
from pmae_tabular.fit_pmae import ProportionalMasker
from pmae_tabular.fit_remasker import ReMasker
from pmae_tabular.evaluate import evaluator

## Load the historical artifacts

In [ ]:
legacy_dir = ROOT / "data" / "legacy" / "diabetes"
proc = torch.load(legacy_dir / "new_proc.pkl", map_location="cpu", weights_only=False)
amputed = torch.load(legacy_dir / "amputed.pkl", map_location="cpu", weights_only=False)
PATTERN = "full"
SEED = 2
experiment = amputed[PATTERN][SEED]
X_true = experiment["X_init"].float()
X_incomplete = experiment["X_incomp"].float()
mask = experiment["mask"].bool()
d_numerical = int(proc["d_numerical"])
miss_cols = [i for i in range(X_true.shape[1]) if i not in experiment["full_cols"]]
print("Shape:", tuple(X_true.shape), "numerical columns:", d_numerical)

## Train the models

Set `EPOCHS=300` for the original-length training run. Use fewer epochs only for a software smoke test.

In [ ]:
EPOCHS = 300
args = _namespace(epochs=EPOCHS, batch_size=128, device="cpu",
                  embed_dim=32, depth=4, decoder_depth=2, num_heads=4)

pmae = ProportionalMasker(args)
X_pmae = pmae.fit(pd.DataFrame(X_incomplete.numpy()))

remasker = ReMasker(args)
X_remasker = remasker.fit(pd.DataFrame(X_incomplete.numpy()))

## Evaluate only the artificially hidden entries

In [ ]:
pmae_scores = evaluator(X_true, X_pmae, mask, X_true.shape[1], d_numerical, miss_cols,
                        cat_exists=(X_true.shape[1] > d_numerical))
remasker_scores = evaluator(X_true, X_remasker, mask, X_true.shape[1], d_numerical, miss_cols,
                            cat_exists=(X_true.shape[1] > d_numerical))
pd.DataFrame([pmae_scores, remasker_scores], index=["PMAE", "ReMasker"]).T